# Semantic Python with OpenAI

This notebook demonstrates semantic comparison with ordinary Python syntax. A model call can happen only after a value is explicitly wrapped with `Semantic`.

```python
message = Semantic("I'm done")

if message == "the user wants to stop":
    print("Stop requested")
```

This is a **live OpenAI demo**. The first semantic comparison sends the complete wrapped state and proposition to OpenAI and may use API credit. Identical later comparisons are served from the in-memory cache.

## 1. Enter the API key securely

Run this cell and paste the key into the hidden prompt. The key is kept in kernel memory; it is not written into this notebook. You must enter it again after restarting the kernel.

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

print("API key configured:", bool(os.environ.get("OPENAI_API_KEY")))

## 2. Configure the OpenAI backend

Configuration does not call the API. The call happens when a `Semantic` value is compared with a proposition string.

The conservative truth policy below accepts probabilities at or above `0.85` as true and at or below `0.15` as false. A result between those thresholds raises `UncertainDecisionError` instead of silently becoming `False`.

In [ ]:
from semantic_python import OpenAIBackend, Semantic, UncertainDecisionError, configure

MODEL = "gpt-6-luna"

configure(
    backend=OpenAIBackend(model=MODEL),
    true_threshold=0.85,
    false_threshold=0.15,
    uncertainty="raise",
    cache=True,
)

print(f"Configured OpenAI model: {MODEL}")

## 3. Make one live semantic decision

This cell makes the notebook's one new OpenAI request. The model receives both `state` and `proposition`. The result is a `Decision`, not a bare boolean, so probability and provenance remain available for inspection.

In [ ]:
state = "I'm finished with this task and want to stop."
proposition = "the user wants to stop"
message = Semantic(state)

decision = message == proposition

print("Value:", decision.value)
print("Probability proposition is true:", decision.probability)
print("Backend:", decision.backend)
print("Model:", decision.model)
print("Response ID:", decision.metadata.get("response_id"))
print("Confidence kind:", decision.metadata.get("confidence_kind"))
print("State digest:", decision.state_hash[:16] + "…")
print("Returned from cache:", decision.cached)

## 4. Resolve the decision safely

Python asks the decision for its truth value inside `if`. If the probability falls inside the configured uncertainty band, the library raises an explicit error so the application can request human review.

The comparison below is identical to the previous one, so it reuses the cached decision and does not make another OpenAI request.

In [ ]:
try:
    if message == proposition:
        print("Action: stop the task")
    else:
        print("Action: continue the task")
except UncertainDecisionError as error:
    print("Action: ask a human to review")
    print("Probability:", error.decision.probability)

## 5. Negation and caching

`!=` negates the same semantic judgment. It does not ask the model a separately worded question. The probability is transformed mechanically from `p` to `1 - p`, and provenance is preserved.

In [ ]:
equal_again = message == proposition
not_equal = message != proposition

print("Equality cached:", equal_again.cached)
print("Equality:", equal_again.value, equal_again.probability)
print("Inequality:", not_equal.value, not_equal.probability)
print("Negated from:", not_equal.metadata.get("negated_from"))

## What to remember

- Ordinary Python values remain ordinary. Only `Semantic(...)` values can invoke a model.
- A comparison sends the complete wrapped state and proposition to the configured OpenAI backend. Do not wrap secrets or data you are not permitted to transfer.
- Model-reported probability is not truth and is not guaranteed to be calibrated for your application.
- Backend failures and uncertain decisions raise errors unless you explicitly configure another policy.
- The in-memory cache reuses only identical decisions and disappears when the Python process ends.
- Restart the kernel when finished if you want to remove the API key from memory.

The backend uses the OpenAI Responses API with Structured Outputs so every successful response follows the required decision schema. See the [official OpenAI Structured Outputs guide](https://developers.openai.com/api/docs/guides/structured-outputs?api-mode=responses).